---
## 1. Conversión de un real a float32

| campo | bits | posición |
|---|---|---|
| signo $s$ | 1 | 0 |
| exponente  $E$ | 8 | 1–8 |
| fracción $F$ | 23 | 9–31 |


$$nim= (-1)^s \cdot \left(1 + \frac{F}{2^{23}}\right) \cdot 2^{\,E-127}$$



### Algoritmo

1. Separar el signo y trabajar con el valor absoluto
2. Escribo el numero como $ num= m \cdot 2^e $ con $1 \le m < 2$ (lo hace `math.frexp`).
3. transformar el exponente: $E = e + 127$.
4. Fraccion $F = \mathrm{round}\big((m-1)\cdot 2^{23}\big)$. Se redondea porque en general
   $x$ **no** es representable exacto en 32 bits.
5. Escribir $s$, $E$ y $F$ en binario y concatenar.

In [14]:
import math

def float32_bits(x):
    x = float(x)
    s = 1 if x < 0 else 0
    a = abs(x)

    if a == 0.0:
        return [0] * 32

    m, e = math.frexp(a)          # a = m · 2^e  con  0.5 <= m < 1
    m, e = m * 2, e - 1           # a = m · 2^e  con  1 <= m < 2

    E = e + 127                   # exponente sesgado, 8 bits
    F = round((m - 1) * 2**23)    # Fraccion sin el 1
    
    bits = f"{s}{E:08b}{F:023b}"  # string de 32 caracteres
    return [int(b) for b in bits]

In [13]:
float32_bits(1.4)

[0,
 0,
 1,
 1,
 1,
 1,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 1]

---
## 2. Precisión de la máquina

El $\varepsilon$ es el número más pequeño tal que

$$0 + \varepsilon \neq 0$$

en la aritmética del computador. 

La idea del copdigo: partir de $\varepsilon = 1$ e ir dividiendo por 2 mientras la suma
$0+\varepsilon$ siga siendo distinta de 1. El último valor que "se nota" es el epsilon.

In [16]:
def precision_maquina(num=0, No_imprimo=False):
    precision = 1
    n = 0
    while num + precision / 2 != num:     # divido en 2 lo que le subo y reviso
        precision = precision / 2
        n += 1
        if No_imprimo and n % 100 == 0:
            print(f"  iteración {n:3d}:  precision = {precision:.5e}")
    return precision


precision = precision_maquina(No_imprimo=True)
print(precision)
print(f"comprobación:  1 + precision   != 1  ->  {1 + precision != 1}")
print(f"comprobación:  1 + precision/2 == 1  ->  {1 + precision/2 == 1}")

  iteración 100:  precision = 7.88861e-31
  iteración 200:  precision = 6.22302e-61
  iteración 300:  precision = 4.90909e-91
  iteración 400:  precision = 3.87259e-121
  iteración 500:  precision = 3.05494e-151
  iteración 600:  precision = 2.40992e-181
  iteración 700:  precision = 1.90109e-211
  iteración 800:  precision = 1.49970e-241
  iteración 900:  precision = 1.18305e-271
  iteración 1000:  precision = 9.33264e-302
5e-324
comprobación:  1 + precision   != 1  ->  False
comprobación:  1 + precision/2 == 1  ->  True
